<a href="https://colab.research.google.com/github/KimDSunny/Merry-Christmas/blob/main/03_humanoid_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Cell 1: Environment Setup ===
# CRITICAL: Set MUJOCO_GL to "egl" BEFORE any mujoco/gymnasium import.
# Colab has no display, so MuJoCo must use the EGL headless backend.
import os
os.environ["MUJOCO_GL"] = "egl"

!pip install -q "gymnasium[mujoco]" "stable-baselines3==2.6.0" "imageio[ffmpeg]"

import torch, gymnasium as gym
import stable_baselines3 as sb3

print("CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print(f"gymnasium: {gym.__version__} | sb3: {sb3.__version__}")
print(f"MUJOCO_GL: {os.environ.get('MUJOCO_GL')}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


CUDA: True - Tesla T4
gymnasium: 1.1.1 | sb3: 2.6.0
MUJOCO_GL: egl


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# === Cell 2: Drive Mount + Checkpoint Paths ===
# Mount Google Drive so models survive VM restarts.
# All checkpoints will be saved under /content/drive/MyDrive/ufb-ghost/

from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/ufb-ghost/checkpoints'
VIDEO_DIR = '/content/drive/MyDrive/ufb-ghost/videos'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

print(f'CKPT_DIR : {CKPT_DIR}')
print(f'VIDEO_DIR: {VIDEO_DIR}')
print('existing checkpoints:', os.listdir(CKPT_DIR) or '(none yet)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CKPT_DIR : /content/drive/MyDrive/ufb-ghost/checkpoints
VIDEO_DIR: /content/drive/MyDrive/ufb-ghost/videos
existing checkpoints: ['walker2d_50000_steps.zip', 'walker2d_100000_steps.zip', 'walker2d_150000_steps.zip', 'walker2d_200000_steps.zip', 'walker2d_ppo.zip']


In [2]:
# === Cell 3 (rev): Memory monitoring + N_ENVS=1 ===
# Goal: catch moment of death by logging RAM at every step.

import psutil, time
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

def mem_str(label):
    rss = psutil.Process().memory_info().rss / 1024**3
    vm = psutil.virtual_memory()
    used = vm.used / 1024**3
    total = vm.total / 1024**3
    return f'[{label}] proc:{rss:.2f}GB | sys:{used:.2f}/{total:.2f}GB ({vm.percent:.0f}%)'

print(mem_str('start'))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

ENV_ID = 'HalfCheetah-v4'
N_ENVS = 1  # reduced from 4

env = DummyVecEnv([lambda: gym.make(ENV_ID) for _ in range(N_ENVS)])
print(mem_str('after env'))

model = PPO('MlpPolicy', env, verbose=0, device='cuda')
print(mem_str('after model'))

class MemMonitor(BaseCallback):
    def _on_step(self):
        if self.num_timesteps % 1000 == 0:
            print(mem_str(f'step {self.num_timesteps}'))
        return True

print()
print(f'Training {ENV_ID}: 10k steps with N_ENVS={N_ENVS}')
t0 = time.time()
model.learn(total_timesteps=10_000, callback=MemMonitor())
print(mem_str('after learn'))
print(f'Done in {time.time()-t0:.0f}s')

[start] proc:2.13GB | sys:2.23/12.67GB (20%)
161 MiB, 15360 MiB
[after env] proc:2.13GB | sys:2.21/12.67GB (20%)
[after model] proc:2.13GB | sys:2.21/12.67GB (20%)

Training HalfCheetah-v4: 10k steps with N_ENVS=1


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:519: DeprecationWarning: WARN: The environment HalfCheetah-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


[step 1000] proc:2.13GB | sys:2.19/12.67GB (20%)
[step 2000] proc:2.13GB | sys:2.20/12.67GB (20%)
[step 3000] proc:2.13GB | sys:2.20/12.67GB (20%)
[step 4000] proc:2.13GB | sys:2.21/12.67GB (20%)
[step 5000] proc:2.13GB | sys:2.21/12.67GB (20%)
[step 6000] proc:2.13GB | sys:2.23/12.67GB (20%)
[step 7000] proc:2.13GB | sys:2.23/12.67GB (20%)
[step 8000] proc:2.13GB | sys:2.23/12.67GB (20%)
[step 9000] proc:2.13GB | sys:2.23/12.67GB (20%)
[step 10000] proc:2.13GB | sys:2.19/12.67GB (20%)
[after learn] proc:2.13GB | sys:2.19/12.67GB (20%)
Done in 22s


In [ ]:
# === Cell: Video render (fresh kernel, load model from disk) ===
from stable_baselines3 import PPO
import gymnasium as gym
import imageio
from IPython.display import HTML
from base64 import b64encode

ENV_ID = 'Walker2d-v4'
CKPT_PATH = f'{CKPT_DIR}/walker2d_ppo'  # define here in case Cell 4 not re-run
model = PPO.load(CKPT_PATH)
print('model loaded:', CKPT_PATH)

render_env = gym.make(ENV_ID, render_mode='rgb_array')
obs, _ = render_env.reset()
frames = []
for _ in range(1000):
    frames.append(render_env.render())
    action, _ = model.predict(obs, deterministic=True)
    obs, r, done, trunc, _ = render_env.step(action)
    if done or trunc:
        break
print(f'frames: {len(frames)}')

video_path = f'{VIDEO_DIR}/walker2d.mp4'
imageio.mimsave(video_path, frames, fps=30, quality=8)
print(f'saved: {video_path}')
mp4 = open(video_path, 'rb').read()
HTML(f'<video width=480 controls autoplay loop><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:519: DeprecationWarning: WARN: The environment Walker2d-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


model loaded: /content/drive/MyDrive/ufb-ghost/checkpoints/walker2d_ppo


In [3]:
# === Cell: Video render (fresh kernel, load model from disk) ===
from stable_baselines3 import PPO
import gymnasium as gym
import imageio
from IPython.display import HTML
from base64 import b64encode

ENV_ID = 'Walker2d-v4'
model = PPO.load(CKPT_PATH)
print('model loaded:', CKPT_PATH)

render_env = gym.make(ENV_ID, render_mode='rgb_array')
obs, _ = render_env.reset()
frames = []
for _ in range(1000):
    frames.append(render_env.render())
    action, _ = model.predict(obs, deterministic=True)
    obs, r, done, trunc, _ = render_env.step(action)
    if done or trunc:
        break
print(f'frames: {len(frames)}')

video_path = f'{VIDEO_DIR}/walker2d.mp4'
imageio.mimsave(video_path, frames, fps=30, quality=8)
print(f'saved: {video_path}')
mp4 = open(video_path, 'rb').read()
HTML(f'<video width=480 controls autoplay loop><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

NameError: name 'CKPT_PATH' is not defined